# Compute Coverage Pipeline

Reads isochrone polygons and census tract geometries,
computes the proportional area overlap between each
branch isochrone and each census tract.

Processes all modes and travel times present in the isochrone cache.

In [ ]:
import os
import kfp
from kfp import dsl, kubernetes
from kfp.dsl import Output, Dataset

## Pipeline Component

In [ ]:
RUNTIME_IMAGE = "image-registry.openshift-image-registry.svc:5000/redhat-ods-applications/runtime-datascience:datascience"


@dsl.component(
    base_image=RUNTIME_IMAGE,
    packages_to_install=["boto3", "shapely", "pandas", "pyarrow", "pyproj", "geopandas"],
    pip_index_urls=["https://pypi.org/simple"],
)
def compute_coverage(
    coverage_out: Output[Dataset],
):
    import io
    import os
    import logging
    import pandas as pd
    import geopandas as gpd
    import boto3
    from shapely import wkt
    from pyproj import Geod

    logging.basicConfig(level=logging.INFO)
    logger = logging.getLogger("compute_coverage")

    geod = Geod(ellps="WGS84")

    s3 = boto3.client(
        "s3",
        endpoint_url=os.environ["AWS_S3_ENDPOINT"],
        aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
        aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
        verify=False,
    )
    bucket = os.environ["AWS_S3_BUCKET"]

    # Load census tracts
    obj = s3.get_object(Bucket=bucket, Key="clean/population_demographics.csv")
    tracts = pd.read_csv(io.BytesIO(obj["Body"].read()))
    tracts["geometry"] = tracts["geometry"].apply(wkt.loads)
    tracts = gpd.GeoDataFrame(tracts, geometry="geometry", crs="EPSG:4326")
    logger.info(f"Loaded {len(tracts)} census tracts")

    # Compute tract areas
    tract_areas = []
    for _, row in tracts.iterrows():
        area, _ = geod.geometry_area_perimeter(row["geometry"])
        tract_areas.append(abs(area))
    tracts["area_sqm"] = tract_areas

    # Load isochrones — use latest per (branch, mode, travel_time)
    obj = s3.get_object(Bucket=bucket, Key="features/isochrones.parquet")
    iso_df = pd.read_parquet(io.BytesIO(obj["Body"].read()))
    iso_df["event_timestamp"] = pd.to_datetime(iso_df["event_timestamp"], utc=True)

    idx = iso_df.groupby(["branch", "mode", "travel_time"])["event_timestamp"].idxmax()
    iso_df = iso_df.loc[idx].reset_index(drop=True)
    logger.info(
        f"Using {len(iso_df)} isochrones across "
        f"{iso_df['mode'].nunique()} modes and "
        f"{iso_df['travel_time'].nunique()} travel times"
    )

    # Compute intersections
    rows = []
    for i, iso in iso_df.iterrows():
        iso_geom = wkt.loads(iso["geometry_wkt"])
        branch = iso["branch"]
        mode = iso["mode"]
        tt = iso["travel_time"]

        for j, tract in tracts.iterrows():
            if iso_geom.intersects(tract["geometry"]):
                intersection = iso_geom.intersection(tract["geometry"])
                inter_area, _ = geod.geometry_area_perimeter(intersection)
                inter_area = abs(inter_area)
                overlap = inter_area / tract["area_sqm"] if tract["area_sqm"] > 0 else 0

                rows.append({
                    "geoid": tract["geoid"],
                    "branch": branch,
                    "mode": mode,
                    "travel_time": tt,
                    "overlap_proportion": overlap,
                    "intersection_wkt": intersection.wkt,
                })

        if (i + 1) % 25 == 0:
            logger.info(f"Processed {i + 1}/{len(iso_df)} isochrones")

    coverage = pd.DataFrame(rows)

    # Write to S3
    buf = io.BytesIO()
    coverage.to_parquet(buf, index=False)
    buf.seek(0)
    s3.put_object(Bucket=bucket, Key="results/coverage_details.parquet", Body=buf.getvalue())

    coverage.to_parquet(coverage_out.path, index=False)
    logger.info(f"Computed {len(coverage)} tract-branch intersections")

## Pipeline Definition

In [ ]:
@dsl.pipeline(
    name="cpl-compute-coverage",
    description="Compute library coverage scores from isochrones and census tracts",
)
def compute_coverage_pipeline():
    task = compute_coverage()
    kubernetes.use_secret_as_env(
        task,
        secret_name="cpl-s3",
        secret_key_to_env={
            "AWS_ACCESS_KEY_ID": "AWS_ACCESS_KEY_ID",
            "AWS_SECRET_ACCESS_KEY": "AWS_SECRET_ACCESS_KEY",
            "AWS_S3_ENDPOINT": "AWS_S3_ENDPOINT",
            "AWS_S3_BUCKET": "AWS_S3_BUCKET",
        },
    )

## Submit

In [ ]:
import subprocess
from kfp.compiler import Compiler
from kfp_server_api.exceptions import ApiException

token = subprocess.check_output("oc whoami -t", shell=True, text=True).strip()
pipeline_url = os.environ.get("PIPELINES_URL")

client = kfp.Client(
    host=pipeline_url,
    existing_token=token,
    verify_ssl=False,
)

Compiler().compile(compute_coverage_pipeline, "compute_coverage_pipeline.yaml")

try:
    client.upload_pipeline("compute_coverage_pipeline.yaml", pipeline_name="cpl-compute-coverage")
    print("Pipeline definition registered.")
except ApiException as e:
    if e.status == 409:
        print("Pipeline definition already exists.")
    else:
        raise

run = client.create_run_from_pipeline_func(
    compute_coverage_pipeline,
    experiment_name="cpl-coverage",
    run_name="compute-coverage-run",
    enable_caching=False,
)
print(f"Pipeline run submitted: {run.run_id}")